In [1]:
import sys
sys.path.insert(0, "/Users/cowboy/GitHub/geosaurus/src")

In [13]:
import warnings
warnings.filterwarnings('ignore')

# Importing, Exporting, Geometry Engines, Oh My!

Now that we've seen the basics of Pandas, NumPy, and the Spatially Enabled DataFrame, let's cover one of the most useful applications of it: reading and writing data. In addition to manipulation and analysis, the Spatially Enabled DataFrame gives you a wide range of options for handling and persisting data. We'll cover the basics of how it works, and some new functionality added in the ArcGIS API for Python version 2.4.1 that gives users even more power in the ways they can read/write.

### Behind the Scenes

First, let's go over how things work behind the scenes. As I'm sure many of you are familiar with, there are numerous different formats that spatial data can be stored in. The two most common ones are File Geodatabase and Shapefile, which are the ones we'll focus on here. These two formats were both designed by Esri, and went in tandem with or resulted in the development of geometry engines that could process these respective types. Most notably, ArcPy is used for File Geodatabases, and shapely/pyshp for Shapefiles. This meant that prior to 2.4.1, users had to have arcpy in their environment to read/write File Geodatabases, and a Shapefile-specific engine to read and write Shapefile. However, we've brought some changes.

We've worked GDAL into the reading/writing functions behind the scenes, which means that users can now easily translate these data types with one single shape engine. For anybody unfamiliar, GDAL stands for Geospatial Data Abstraction Library, and is an open source library that can read/write a wide variety of raster and vector geospatial data formats. It already comes in the standard environment for ArcGIS Pro on Windows, and can also be installed via Python package managers Conda and Pip on Linux & MacOS. This means that File Geodatabases are now easily read into/written from the SeDF on Linux and Mac. Let's jump in!

In [3]:
# set the environment variable with os.environ
# very important to do this before importing arcgis! must reset environment otherwise
import os
os.environ["ARCGIS_GEOMETRY_ENGINE"] = "gdal" # 'gdal', 'shapefile', 'arcpy'

In [4]:
from arcgis.features import GeoAccessor, GeoSeriesAccessor
import pandas as pd

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [5]:
# these are hidden properties that show us environment info
from arcgis._impl._geometry_engine import SELECTED_ENGINE, HAS_ARCPY, HAS_GDAL, HAS_PYSHP
print("Has ArcPy: " + str(HAS_ARCPY))
print("Has GDAL: " + str(HAS_GDAL))
print("Has PySHP: " + str(HAS_PYSHP))
print("Selected Engine: " + str(SELECTED_ENGINE))

Has ArcPy: False
Has GDAL: True
Has PySHP: True
Selected Engine: GeometryEngine.GDAL


#### Shapefile

In [7]:
# importing from a shapefile on disk
sedf = pd.DataFrame.spatial.from_featureclass(r"/Users/cowboy/Downloads/volcano_status/volcano_status.shp")
sedf

,volcano_na,observatio,alert_leve,color_code,region,alert_date,threat,link,latitude,longitude,object_id,SHAPE
0,Buldir,avo,UNASSIGNED,UNASSIGNED,Aleutians,NaT,Low Threat,https://avo.alaska.edu/volcanoes/volcinfo.php?...,52.3488,175.9090,1611,"{""x"": 19582100.305953957, ""y"": 6863440.3533529..."
1,Chagulak,avo,UNASSIGNED,UNASSIGNED,Aleutians,NaT,Low Threat,https://avo.alaska.edu/volcanoes/volcinfo.php?...,52.5714,-171.1395,1612,"{""x"": -19051161.994615443, ""y"": 6904108.706134..."
2,Cinnamon Butte,cvo,UNASSIGNED,UNASSIGNED,Oregon,NaT,Very Low Threat,https://www.usgs.gov/volcanoes/cinnamon-butte,43.2410,-122.1080,1613,"{""x"": -13593000.38178505, ""y"": 5348726.7434948..."
3,Cleveland,avo,NORMAL,GREEN,Aleutians,2024-07-08,Moderate Threat,https://avo.alaska.edu/activity/Cleveland.php,52.8222,-169.9450,1614,"{""x"": -18918190.862862874, ""y"": 6950177.212884..."
4,Coso Volcanic Field,calvo,NORMAL,GREEN,California,2019-07-11,Moderate Threat,https://www.usgs.gov/volcanoes/coso-volcanic-f...,36.0300,-117.8200,1615,"{""x"": -13115662.405263491, ""y"": 4304750.111338..."
...,...,...,...,...,...,...,...,...,...,...,...,...
156,Tanaga,avo,NORMAL,GREEN,Aleutians,2024-01-29,Moderate Threat,https://avo.alaska.edu/activity/Tanaga.php,51.8840,-178.1430,1767,"{""x"": -19830788.048386134, ""y"": 6779178.281346..."
157,Veniaminof,avo,NORMAL,GREEN,Alaska Peninsula,2021-07-09,High Threat,https://avo.alaska.edu/activity/Veniaminof.php,56.1979,-159.3931,1768,"{""x"": -17743558.727961335, ""y"": 7597913.242954..."
158,Wapi Lava Field,yvo,UNASSIGNED,UNASSIGNED,Idaho,NaT,Low Threat,https://www.usgs.gov/volcanoes/wapi-lava-field,42.8800,-113.2200,1769,"{""x"": -12603592.747614434, ""y"": 5293724.423322..."
159,Yellowstone,yvo,NORMAL,GREEN,Wyoming,2025-03-01,High Threat,https://www.usgs.gov/volcanoes/yellowstone,44.4300,-110.6700,1770,"{""x"": -12319728.046091586, ""y"": 5532228.609532..."


#### File Geodatabase

In [8]:
# importing from fgdb on disk
sedf = pd.DataFrame.spatial.from_featureclass(r"/Users/cowboy/Downloads/volcano_status.gdb")
sedf

,volcano_name,observation,alert_level,color_code,region,alert_date,threat,link,latitude,longitude,object_id,SHAPE
0,Buldir,avo,UNASSIGNED,UNASSIGNED,Aleutians,None,Low Threat,https://avo.alaska.edu/volcanoes/volcinfo.php?...,52.3488,175.9090,1611.0,"{""x"": 19582100.306000233, ""y"": 6863440.3534002..."
1,Chagulak,avo,UNASSIGNED,UNASSIGNED,Aleutians,None,Low Threat,https://avo.alaska.edu/volcanoes/volcinfo.php?...,52.5714,-171.1395,1612.0,"{""x"": -19051161.994600058, ""y"": 6904108.706099..."
2,Cinnamon Butte,cvo,UNASSIGNED,UNASSIGNED,Oregon,None,Very Low Threat,https://www.usgs.gov/volcanoes/cinnamon-butte,43.2410,-122.1080,1613.0,"{""x"": -13593000.381799936, ""y"": 5348726.743500..."
3,Cleveland,avo,NORMAL,GREEN,Aleutians,2024-07-08 21:06:48+00,Moderate Threat,https://avo.alaska.edu/activity/Cleveland.php,52.8222,-169.9450,1614.0,"{""x"": -18918190.86290002, ""y"": 6950177.2129001..."
4,Coso Volcanic Field,calvo,NORMAL,GREEN,California,2019-07-11 23:36:44+00,Moderate Threat,https://www.usgs.gov/volcanoes/coso-volcanic-f...,36.0300,-117.8200,1615.0,"{""x"": -13115662.405299902, ""y"": 4304750.111299..."
...,...,...,...,...,...,...,...,...,...,...,...,...
156,Tanaga,avo,NORMAL,GREEN,Aleutians,2024-01-29 23:13:47+00,Moderate Threat,https://avo.alaska.edu/activity/Tanaga.php,51.8840,-178.1430,1767.0,"{""x"": -19830788.048399925, ""y"": 6779178.281300..."
157,Veniaminof,avo,NORMAL,GREEN,Alaska Peninsula,2021-07-09 21:30:29+00,High Threat,https://avo.alaska.edu/activity/Veniaminof.php,56.1979,-159.3931,1768.0,"{""x"": -17743558.727999926, ""y"": 7597913.243000..."
158,Wapi Lava Field,yvo,UNASSIGNED,UNASSIGNED,Idaho,None,Low Threat,https://www.usgs.gov/volcanoes/wapi-lava-field,42.8800,-113.2200,1769.0,"{""x"": -12603592.747600079, ""y"": 5293724.423299..."
159,Yellowstone,yvo,NORMAL,GREEN,Wyoming,2025-03-01 15:39:40+00,High Threat,https://www.usgs.gov/volcanoes/yellowstone,44.4300,-110.6700,1770.0,"{""x"": -12319728.046099901, ""y"": 5532228.609499..."


#### Feature Layer

In [9]:
# importing from feature layer
from arcgis.gis import GIS
from arcgis.features import FeatureLayer
gis = GIS(profile = "your_online_profile")
flayer = FeatureLayer("https://services7.arcgis.com/JEwYeAy2cc8qOe3o/arcgis/rest/services/kc_house_data_XYTableToPoint/FeatureServer/0")
sedf = pd.DataFrame.spatial.from_layer(flayer)
sedf

,OBJECTID,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,...,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,SHAPE
0,201,3542300060.0,20150311T000000,210000.0,3,1.0,860,11725,1.0,0,...,860,0,1943,0,98056,47.5093,-122.184,1300,9514,"{""x"": -122.18399999999997, ""y"": 47.50930000093..."
1,202,2222059065.0,20141112T000000,297000.0,3,2.5,1940,14952,2.0,0,...,1940,0,1994,0,98042,47.3777,-122.165,2030,10450,"{""x"": -122.16499999999995, ""y"": 47.37770000093..."
2,203,7551300060.0,20140716T000000,470000.0,3,1.0,1010,5000,1.0,0,...,1010,0,1952,0,98107,47.675,-122.394,1680,5000,"{""x"": -122.39399999999996, ""y"": 47.67500000093..."
3,204,100600550.0,20140804T000000,226500.0,3,1.5,1300,7370,1.0,0,...,900,400,1976,0,98023,47.3025,-122.37,1430,7500,"{""x"": -122.36999999999995, ""y"": 47.30250000094..."
4,205,3211100860.0,20150303T000000,274250.0,3,1.0,910,8450,1.0,0,...,910,0,1962,0,98059,47.4787,-122.158,1400,8040,"{""x"": -122.15799999999996, ""y"": 47.47870000093..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21608,21609,263000018.0,20140521T000000,360000.0,3,2.5,1530,1131,3.0,0,...,1530,0,2009,0,98103,47.6993,-122.346,1530,1509,"{""x"": -122.34599999999996, ""y"": 47.69930000093..."
21609,21610,6600060120.0,20150223T000000,400000.0,4,2.5,2310,5813,2.0,0,...,2310,0,2014,0,98146,47.5107,-122.362,1830,7200,"{""x"": -122.36199999999997, ""y"": 47.51070000093..."
21610,21611,1523300141.0,20140623T000000,402101.0,2,0.75,1020,1350,2.0,0,...,1020,0,2009,0,98144,47.5944,-122.299,1020,2007,"{""x"": -122.29899999999999, ""y"": 47.59440000093..."
21611,21612,291310100.0,20150116T000000,400000.0,3,2.5,1600,2388,2.0,0,...,1600,0,2004,0,98027,47.5345,-122.069,1410,1287,"{""x"": -122.06899999999995, ""y"": 47.53450000093..."


#### URL

In [ ]:
# importing from shapefile at a URL
sedf = pd.DataFrame.spatial.from_featureclass(r"https://www.arcgis.com/sharing/rest/content/items/804abf2794b346828eeff285bffe9259/data")
sedf

#### CSV

In [11]:
# read a CSV into a dataframe then use from_xy
df = pd.read_csv(r"/Users/cowboy/Downloads/new.csv")
sedf = pd.DataFrame.spatial.from_xy(df, "x", "y")
sedf

,FID,city_id,name,state,capital,pop2000,pop2007,longitude,latitude,ObjectId,x,y,SHAPE
0,1,1,Honolulu,HI,State,37165,37858,-157.823436,21.305782,1,-157.823436,21.305782,"{""spatialReference"": {""wkid"": 4326}, ""x"": -157..."
1,2,2,Juneau,AK,State,30711,31592,-134.511582,58.351418,2,-134.511582,58.351418,"{""spatialReference"": {""wkid"": 4326}, ""x"": -134..."
2,3,3,Boise City,ID,State,18578,20352,-116.237655,43.613736,3,-116.237655,43.613736,"{""spatialReference"": {""wkid"": 4326}, ""x"": -116..."
3,4,4,Olympia,WA,State,27514,45523,-122.893073,47.042418,4,-122.893073,47.042418,"{""spatialReference"": {""wkid"": 4326}, ""x"": -122..."
4,5,5,Salem,OR,State,13692,15203,-123.029155,44.931109,5,-123.029155,44.931109,"{""spatialReference"": {""wkid"": 4326}, ""x"": -123..."
5,6,6,Carson,NV,State,52457,56641,-119.753873,39.160946,6,-119.753873,39.160946,"{""spatialReference"": {""wkid"": 4326}, ""x"": -119..."
6,7,7,Sacramento,CA,State,40701,46291,-121.468927,38.555609,7,-121.468927,38.555609,"{""spatialReference"": {""wkid"": 4326}, ""x"": -121..."
7,8,8,Phoenix,AZ,State,13210,15021,-112.076300,33.528373,8,-112.076300,33.528373,"{""spatialReference"": {""wkid"": 4326}, ""x"": -112..."
8,9,9,Salt Lake City,UT,State,18174,18536,-111.892618,40.754700,9,-111.892618,40.754700,"{""spatialReference"": {""wkid"": 4326}, ""x"": -111..."
9,10,10,Cheyenne,WY,State,53011,54750,104.802046,-41.145545,10,104.802046,-41.145545,"{""spatialReference"": {""wkid"": 4326}, ""x"": 104...."


### Writing Out

Now that we have more SeDF's than we know what to do with, let's spam our local disk and GIS org with them in different ways!

In [ ]:
flayer = FeatureLayer("https://services7.arcgis.com/JEwYeAy2cc8qOe3o/arcgis/rest/services/kc_house_data_XYTableToPoint/FeatureServer/0")
sedf = pd.DataFrame.spatial.from_layer(flayer)

In [ ]:
# first let's do to shapefile
sedf.spatial.to_featureclass(r"/Users/cowboy/Workspace/DevSummit2025/dataframes/shape_data/shape_data.shp")

In [ ]:
# then let's do to file geodatabase
sedf.spatial.to_featureclass(r"/Users/cowboy/Workspace/DevSummit2025/dataframes/fgdb_data.gdb")